# T34 - 债券新发行

## 第6章：测试与验证

---

### 本章概述

本章实现债券新发行系统的测试和验证功能，包括：
1. 单元测试
2. 数据完整性验证
3. 数据质量检查
4. 功能测试

### 导入依赖

In [ ]:
# 标准库
import os
import sys
import datetime
from datetime import datetime, timedelta
import warnings

# 数据处理
import pandas as pd
import numpy as np

# 数据库
import sqlalchemy
from sqlalchemy import text, create_engine
import sqlalchemy.pool as pool

# 忽略警告
warnings.filterwarnings('ignore')

print("依赖库导入成功")

### 数据库连接

In [ ]:
# 数据库配置
MYSQL_HOST = os.environ.get('MYSQL_HOST', 'rm-uf6c8yi6zdq6ea7p1qo.mysql.rds.aliyuncs.com')
MYSQL_PORT = os.environ.get('MYSQL_PORT', '3306')
MYSQL_USER = os.environ.get('MYSQL_USER', 'hz_work')
MYSQL_PASSWORD = os.environ.get('MYSQL_PASSWORD', '')
MYSQL_DATABASE = os.environ.get('MYSQL_DATABASE', 'yq')

def get_db_engine():
    connection_string = f'mysql+pymysql://{MYSQL_USER}:{MYSQL_PASSWORD}@{MYSQL_HOST}:{MYSQL_PORT}/{MYSQL_DATABASE}'
    return create_engine(connection_string, poolclass=pool.NullPool, pool_recycle=3600)

sql_engine = get_db_engine()
print("数据库连接创建成功")

### 数据完整性验证

In [ ]:
class DataValidator:
    """数据验证器"""
    
    def __init__(self, engine):
        self.engine = engine
    
    def check_table_exists(self, table_name: str) -> bool:
        """检查表是否存在"""
        try:
            with self.engine.connect() as conn:
                result = conn.execute(text(f"SHOW TABLES LIKE '{table_name}'"))
                return result.fetchone() is not None
        except Exception as e:
            print(f"检查表存在失败: {e}")
            return False
    
    def get_table_row_count(self, table_name: str) -> int:
        """获取表行数"""
        try:
            with self.engine.connect() as conn:
                result = conn.execute(text(f"SELECT COUNT(*) FROM `{table_name}`"))
                return result.fetchone()[0]
        except Exception as e:
            print(f"获取行数失败: {e}")
            return 0
    
    def check_date_coverage(self, table_name: str, date_column: str = 'dt', 
                            days: int = 7) -> dict:
        """
        检查日期覆盖范围
        
        Args:
            table_name: 表名
            date_column: 日期列名
            days: 期望覆盖的天数
            
        Returns:
            日期覆盖统计
        """
        result = {
            'expected_days': days,
            'actual_days': 0,
            'missing_dates': [],
            'date_range': (None, None)
        }
        
        try:
            # 获取日期范围
            current_date = datetime.now()
            start_date = current_date.strftime('%Y-%m-%d')
            end_date = (current_date + timedelta(days=days)).strftime('%Y-%m-%d')
            
            query = f"""
            SELECT DISTINCT {date_column} as dt
            FROM `{table_name}`
            WHERE {date_column} >= '{start_date}'
            ORDER BY {date_column}
            """
            
            with self.engine.connect() as conn:
                df = pd.read_sql(query, conn)
            
            if not df.empty:
                df['dt'] = pd.to_datetime(df['dt'])
                result['actual_days'] = len(df)
                result['date_range'] = (
                    df['dt'].min().strftime('%Y-%m-%d'),
                    df['dt'].max().strftime('%Y-%m-%d')
                )
                
                # 找出缺失的日期
                expected_dates = set(pd.date_range(start=start_date, end=end_date))
                actual_dates = set(df['dt'])
                missing = expected_dates - actual_dates
                result['missing_dates'] = [d.strftime('%Y-%m-%d') for d in sorted(missing)]
        
        except Exception as e:
            print(f"检查日期覆盖失败: {e}")
        
        return result
    
    def check_null_values(self, table_name: str, columns: list) -> dict:
        """
        检查空值
        
        Args:
            table_name: 表名
            columns: 要检查的列名列表
            
        Returns:
            空值统计
        """
        result = {}
        
        for col in columns:
            try:
                query = f"""
                SELECT 
                    COUNT(*) as total,
                    SUM(CASE WHEN `{col}` IS NULL OR `{col}` = '' THEN 1 ELSE 0 END) as null_count
                FROM `{table_name}`
                """
                with self.engine.connect() as conn:
                    row = conn.execute(text(query)).fetchone()
                    result[col] = {
                        'total': row[0],
                        'null_count': row[1],
                        'null_rate': round(row[1] / row[0] * 100, 2) if row[0] > 0 else 0
                    }
            except Exception as e:
                result[col] = {'error': str(e)}
        
        return result
    
    def check_duplicates(self, table_name: str, columns: list) -> int:
        """
        检查重复记录
        
        Args:
            table_name: 表名
            columns: 用于判断重复的列
            
        Returns:
            重复记录数
        """
        col_str = ', '.join([f'`{c}`' for c in columns])
        query = f"""
        SELECT COUNT(*) as dup_count
        FROM (
            SELECT {col_str}, COUNT(*) as cnt
            FROM `{table_name}`
            GROUP BY {col_str}
            HAVING cnt > 1
        ) t
        """
        
        try:
            with self.engine.connect() as conn:
                result = conn.execute(text(query)).fetchone()
                return result[0] if result else 0
        except Exception as e:
            print(f"检查重复失败: {e}")
            return -1


print("DataValidator类定义完成")

### 数据质量检查

In [ ]:
class DataQualityChecker:
    """数据质量检查器"""
    
    def __init__(self, engine):
        self.engine = engine
    
    def check_maturity_data(self) -> dict:
        """检查债券到期数据质量"""
        result = {
            'table': '债券到期',
            'checks': []
        }
        
        validator = DataValidator(self.engine)
        
        # 检查表存在
        exists = validator.check_table_exists('债券到期')
        result['checks'].append({
            'name': '表存在检查',
            'status': 'PASS' if exists else 'FAIL',
            'value': exists
        })
        
        if not exists:
            return result
        
        # 检查行数
        row_count = validator.get_table_row_count('债券到期')
        result['checks'].append({
            'name': '数据量检查',
            'status': 'PASS' if row_count > 0 else 'WARN',
            'value': row_count
        })
        
        # 检查日期覆盖
        date_coverage = validator.check_date_coverage('债券到期', 'dt', 7)
        result['checks'].append({
            'name': '日期覆盖检查',
            'status': 'PASS' if date_coverage['actual_days'] > 0 else 'WARN',
            'value': date_coverage
        })
        
        # 检查空值
        null_stats = validator.check_null_values('债券到期', ['dt', 'totalredemption', 'bondtype'])
        for col, stats in null_stats.items():
            if 'error' not in stats:
                result['checks'].append({
                    'name': f'{col}空值检查',
                    'status': 'PASS' if stats['null_rate'] < 10 else 'WARN',
                    'value': stats
                })
        
        return result
    
    def check_issue_data(self) -> dict:
        """检查债券新发行数据质量"""
        result = {
            'table': '债券新发行1',
            'checks': []
        }
        
        validator = DataValidator(self.engine)
        
        # 检查表存在
        exists = validator.check_table_exists('债券新发行1')
        result['checks'].append({
            'name': '表存在检查',
            'status': 'PASS' if exists else 'FAIL',
            'value': exists
        })
        
        if not exists:
            return result
        
        # 检查行数
        row_count = validator.get_table_row_count('债券新发行1')
        result['checks'].append({
            'name': '数据量检查',
            'status': 'PASS' if row_count > 0 else 'WARN',
            'value': row_count
        })
        
        # 检查空值
        null_stats = validator.check_null_values('债券新发行1', ['trade_code', 'sec_name', 'dt'])
        for col, stats in null_stats.items():
            if 'error' not in stats:
                result['checks'].append({
                    'name': f'{col}空值检查',
                    'status': 'PASS' if stats['null_rate'] == 0 else 'FAIL',
                    'value': stats
                })
        
        # 检查重复
        dup_count = validator.check_duplicates('债券新发行1', ['trade_code'])
        result['checks'].append({
            'name': '债券代码重复检查',
            'status': 'PASS' if dup_count == 0 else 'FAIL',
            'value': dup_count
        })
        
        # 检查债券名称重复（应该已被去重）
        name_dup_count = validator.check_duplicates('债券新发行1', ['sec_name'])
        result['checks'].append({
            'name': '债券名称重复检查',
            'status': 'PASS' if name_dup_count == 0 else 'WARN',
            'value': name_dup_count
        })
        
        return result
    
    def run_all_checks(self) -> dict:
        """运行所有数据质量检查"""
        return {
            'maturity': self.check_maturity_data(),
            'issue': self.check_issue_data(),
            'check_time': datetime.now().strftime('%Y-%m-%d %H:%M:%S')
        }
    
    @staticmethod
    def print_check_result(result: dict):
        """打印检查结果"""
        print(f"\n{'='*60}")
        print(f"表: {result['table']}")
        print('='*60)
        
        for check in result.get('checks', []):
            status = check['status']
            status_str = f"[{status}]"
            
            if status == 'PASS':
                status_str = f"\033[92m{status_str}\033[0m"  # 绿色
            elif status == 'FAIL':
                status_str = f"\033[91m{status_str}\033[0m"  # 红色
            else:
                status_str = f"\033[93m{status_str}\033[0m"  # 黄色
            
            print(f"{status_str} {check['name']}: {check['value']}")


print("DataQualityChecker类定义完成")

### 执行测试

In [ ]:
# 创建检查器
checker = DataQualityChecker(sql_engine)

# 运行所有检查
results = checker.run_all_checks()

# 打印结果
checker.print_check_result(results['maturity'])
checker.print_check_result(results['issue'])

print(f"\n检查时间: {results['check_time']}")

### 数据预览

In [ ]:
# 预览债券到期数据
try:
    query = "SELECT * FROM 债券到期 ORDER BY dt DESC LIMIT 10"
    with sql_engine.connect() as conn:
        df = pd.read_sql(query, conn)
    print("债券到期数据预览:")
    print(df)
except Exception as e:
    print(f"查询失败: {e}")

In [ ]:
# 预览债券新发行数据
try:
    query = "SELECT * FROM 债券新发行1 ORDER BY dt DESC LIMIT 10"
    with sql_engine.connect() as conn:
        df = pd.read_sql(query, conn)
    print("债券新发行数据预览:")
    print(df)
except Exception as e:
    print(f"查询失败: {e}")

### 数据统计

In [ ]:
# 债券到期数据统计
try:
    query = """
    SELECT bondtype, isurbaninvestmentbonds, COUNT(*) as cnt, SUM(totalredemption) as total_amount
    FROM 债券到期
    WHERE dt >= DATE_SUB(CURDATE(), INTERVAL 7 DAY)
    GROUP BY bondtype, isurbaninvestmentbonds
    ORDER BY cnt DESC
    """
    with sql_engine.connect() as conn:
        df = pd.read_sql(query, conn)
    print("债券到期数据统计（近7天）:")
    print(df.to_string())
except Exception as e:
    print(f"查询失败: {e}")

In [ ]:
# 债券新发行数据统计
try:
    query = """
    SELECT bondtype, isurbaninvestmentbonds, COUNT(*) as cnt, SUM(planissueamount) as total_amount
    FROM 债券新发行1
    WHERE dt >= DATE_SUB(CURDATE(), INTERVAL 30 DAY)
    GROUP BY bondtype, isurbaninvestmentbonds
    ORDER BY cnt DESC
    """
    with sql_engine.connect() as conn:
        df = pd.read_sql(query, conn)
    print("债券新发行数据统计（近30天）:")
    print(df.to_string())
except Exception as e:
    print(f"查询失败: {e}")

---

### 本章小结

**已实现功能**:
- `DataValidator`: 数据验证器
- `DataQualityChecker`: 数据质量检查器
- 表存在检查
- 日期覆盖检查
- 空值检查
- 重复记录检查
- 数据预览和统计

---

**下一章**: [7_部署与监控](7_部署与监控.ipynb)